# 간호사 행동/도메인 특화 VLM – LoRA Fine-tuning 과제
## 연구실 서버 (TITAN Xp 12GB) 실행 버전

### 과제 목표
1. **Baseline 분석**: SmolVLM zero-shot caption 생성 및 실패 분석 (의료 object 인식 실패 / 행동 collapse / hallucination)
2. **LoRA Fine-tuning**: 간호 특화 VLM 학습
3. **Before / After 비교**: 정량(BLEU/METEOR/CIDEr-D) + 정성(동일 이미지 시각화)

### 서버 환경
- GPU: NVIDIA TITAN Xp × 4 (각 12GB VRAM) → GPU 0 사용
- 가상환경: `smolvlm_lora`
- 경로: `/home/undergraduate/20221373_YY/aiconvergence1/nurse/`

### 실행 방법 (터미널)
```bash
# tmux 세션으로 백그라운드 실행 (서버 연결 끊겨도 계속 실행)
tmux new -s nurse
conda activate smolvlm_lora
cd ~/20221373_YY/aiconvergence1/nurse
jupyter nbconvert --to notebook --execute nursing_vlm_server.ipynb --output nursing_vlm_result.ipynb
# Ctrl+B, D 로 세션 detach
```

### Caption 작성 규칙
`A nurse {action} using {medical object} in a hospital ward.`

### 선정 Procedure (JSON operationID 기준, 1-indexed)
| operationID | 절차명 | 영상 수 |
|---|---|---|
| 5 | Closed Intravenous Infusion | 60개 |
| 6 | Subcutaneous Injection | 60개 |
| 8 | Blood Glucose Monitoring | 57개 |

> **ID 오프셋 주의**: xlsx는 0-indexed, JSON은 1-indexed → xlsx ID + 1 = JSON ID

---
## 0. 환경 설정

In [1]:
# GPU 고정 — TITAN Xp GPU 3 사용 (GPU 0은 다른 사람 사용 중)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Sun May 31 14:18:31 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA TITAN Xp                Off | 00000000:19:00.0 Off |                  N/A |
| 23%   31C    P8               9W / 250W |     11MiB / 12288MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
# 필요 라이브러리 설치 (가상환경 smolvlm_lora 기준)
# 이미 설치되어 있으면 skip됨
!pip install -q \
    transformers \
    accelerate \
    peft \
    Pillow \
    tqdm \
    nltk \
    pycocoevalcap \
    matplotlib \
    openpyxl

In [3]:
# 공통 import
import json, random, gc, zipfile
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter
from functools import partial

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')   # 서버(display 없음) 환경에서 반드시 필요
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm import tqdm

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet',   quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'사용 device: {DEVICE}')

CUDA available: True
GPU: NVIDIA TITAN Xp
VRAM: 12.8 GB
사용 device: cuda


---
## 1. 경로 설정 및 frames.zip 압축 해제

In [4]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────
BASE_DIR  = Path('/home/undergraduate/20221373_YY/aiconvergence1/nurse')
FRAME_DIR = Path('/data/20221373_YY/nurse/datasets/frames')
MODEL_DIR = Path('/data/20221373_YY/nurse/outputs/model')
ANNO_PATH = BASE_DIR / 'NurViD_annotations.json'
ZIP_PATH  = BASE_DIR / 'frames.zip'

FRAME_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── frames.zip 압축 해제 ──────────────────────────────────────────────────
existing_frames = list(FRAME_DIR.rglob('*.jpg')) if FRAME_DIR.exists() else []

if len(existing_frames) == 0:
    if ZIP_PATH.exists():
        print(f'frames.zip 압축 해제 중: {ZIP_PATH} → {FRAME_DIR}')
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            names = zf.namelist()
            print(f'  zip 내부 파일 수: {len(names)}')
            print(f'  첫 3개: {names[:3]}')
            # zip 내부가 frames/로 시작하면 부모 폴더에, 아니면 FRAME_DIR에 직접 풀기
            target = FRAME_DIR.parent if names[0].startswith('frames/') else FRAME_DIR
            zf.extractall(target)
        print('압축 해제 완료')
    else:
        print(f'❌ frames.zip 없음: {ZIP_PATH}')
else:
    print(f'frames 이미 존재: {len(existing_frames)}개 → 압축 해제 skip')

# ── 경로 확인 ─────────────────────────────────────────────────────────────
all_frames = list(FRAME_DIR.rglob('*.jpg'))
print()
print(f'BASE_DIR  : {BASE_DIR}   {"✅" if BASE_DIR.exists()  else "❌"}')
print(f'ANNO_PATH : {ANNO_PATH}  {"✅" if ANNO_PATH.exists() else "❌"}')
print(f'FRAME_DIR : {FRAME_DIR}  {"✅" if FRAME_DIR.exists() else "❌"}')
print(f'  └ 총 프레임 수: {len(all_frames)}개')
if all_frames:
    print(f'  └ 경로 예시: {all_frames[0]}')

frames 이미 존재: 2060개 → 압축 해제 skip

BASE_DIR  : /home/undergraduate/20221373_YY/aiconvergence1/nurse   ✅
ANNO_PATH : /home/undergraduate/20221373_YY/aiconvergence1/nurse/NurViD_annotations.json  ✅
FRAME_DIR : /data/20221373_YY/nurse/datasets/frames  ✅
  └ 총 프레임 수: 2060개
  └ 경로 예시: /data/20221373_YY/nurse/datasets/frames/XRSI-Yu4NEM/ann00/frame_0001.jpg


---
## 2. 데이터셋 준비

### 2-1. Procedure / Action ID 매핑

> xlsx 0-indexed, JSON 1-indexed → xlsx ID + 1 = JSON ID

In [5]:
# Procedure 매핑 (JSON operationID 기준)
PROCEDURE_MAP = {
    5: 'Closed Intravenous Infusion',
    6: 'Subcutaneous Injection',
    8: 'Blood Glucose Monitoring',
}
TARGET_PROCEDURES = set(PROCEDURE_MAP.keys())

# Action 매핑 (JSON actionID 기준, xlsx 파싱으로 검증 완료)
ACTION_MAP = {
    # op5 Closed IV Infusion
    23: 'Connect infusion device',        # xlsx 22
     8: 'Venipuncture',                   # xlsx  7
     2: 'Disinfect skin',                 # xlsx  1
    95: 'Adjust drip rate',               # xlsx 94
     5: 'Check',                          # xlsx  4
    16: 'Release trapped air',            # xlsx 15
     3: 'Handwashing',                    # xlsx  2
    17: 'Select a vein',                  # xlsx 16
    19: 'Remove needle',                  # xlsx 18
    11: 'Document',                       # xlsx 10
    # op6 Subcutaneous Injection (2,3,5,11,19 위와 공유)
    20: 'Perform subcutaneous puncture',  # xlsx 19
     6: 'Inject medication',              # xlsx  5
    43: 'Aspirate medication',            # xlsx 42
     4: 'Position the patient',           # xlsx  3
    # op8 Blood Glucose Monitoring (2,3,5,11 위와 공유)
    21: 'Measure blood glucose level',    # xlsx 20
    26: 'Prepare glucometer',             # xlsx 25
   117: 'Select needle puncture site',    # xlsx 116
}

print('=== Procedure 매핑 ===')
for pid, name in PROCEDURE_MAP.items():
    print(f'  JSON operationID {pid}: {name}')
print(f'\n총 action 종류: {len(ACTION_MAP)}개')

=== Procedure 매핑 ===
  JSON operationID 5: Closed Intravenous Infusion
  JSON operationID 6: Subcutaneous Injection
  JSON operationID 8: Blood Glucose Monitoring

총 action 종류: 17개


### 2-2. GT Caption 생성 함수

Caption 규칙: **`A nurse {action_phrase} using {medical_object} in a hospital ward.`**

In [6]:
# (operationID, actionID) → (action_phrase, medical_object)
CAPTION_TEMPLATES = {
    # ── op5: Closed IV Infusion ───────────────────────────────────────────
    (5, 23): ('connecting the infusion device to the IV line',
               'an IV infusion set'),
    (5,  8): ("performing venipuncture on the patient's arm",
               'an IV needle and tourniquet'),
    (5,  2): ('disinfecting the skin at the injection site',
               'a disinfectant swab'),
    (5, 95): ('adjusting the IV drip rate',
               'an IV drip chamber'),
    (5,  5): ('checking the patient information and IV fluid bag',
               'an IV fluid bag'),
    (5, 16): ('releasing trapped air from the IV tubing',
               'an IV infusion line'),
    (5,  3): ('performing hand hygiene before IV insertion',
               'soap and running water'),
    (5, 17): ('selecting a suitable vein for IV access',
               'a tourniquet'),
    (5, 19): ('removing the needle after IV catheter insertion',
               'an IV catheter'),
    (5, 11): ('documenting the IV infusion procedure on the patient chart',
               'a patient record sheet'),
    # ── op6: Subcutaneous Injection ───────────────────────────────────────
    (6,  2): ('disinfecting the skin at the subcutaneous injection site',
               'a disinfectant swab'),
    (6, 20): ('performing a subcutaneous puncture on the patient',
               'a syringe and needle'),
    (6,  6): ('injecting medication subcutaneously into the patient',
               'a syringe'),
    (6, 19): ('removing the needle after subcutaneous injection',
               'a syringe and needle'),
    (6, 43): ('aspirating medication into the syringe from a vial',
               'a medication vial and syringe'),
    (6,  3): ('performing hand hygiene before subcutaneous injection',
               'soap and running water'),
    (6, 16): ('releasing trapped air from the syringe',
               'a syringe'),
    (6,  5): ('checking the patient information and medication label',
               'a medication label'),
    (6, 11): ('documenting the subcutaneous injection on the patient chart',
               'a patient record sheet'),
    (6,  4): ('positioning the patient for subcutaneous injection',
               'a hospital bed'),
    # ── op8: Blood Glucose Monitoring ─────────────────────────────────────
    (8, 21): ("measuring the patient's blood glucose level",
               'a glucometer and test strip'),
    (8, 26): ('preparing the glucometer for blood glucose testing',
               'a glucometer and lancet'),
    (8,  2): ('disinfecting the fingertip before blood glucose testing',
               'a disinfectant swab'),
    (8,  3): ('performing hand hygiene before blood glucose monitoring',
               'soap and running water'),
    (8,117): ('selecting the fingertip puncture site for glucose testing',
               'a lancet device'),
    (8, 11): ('documenting the blood glucose result on the patient chart',
               'a patient record sheet'),
    (8,  5): ('checking the patient information before glucose monitoring',
               'a glucometer'),
}


def build_gt_caption(op_id: int, action_id: int) -> str:
    """GT caption 생성: role + action + medical object + hospital context"""
    key = (op_id, action_id)
    if key in CAPTION_TEMPLATES:
        action_phrase, medical_obj = CAPTION_TEMPLATES[key]
    else:
        proc_name   = PROCEDURE_MAP.get(op_id, 'medical procedure')
        action_name = ACTION_MAP.get(action_id, f'action_{action_id}')
        action_phrase = f'performing {action_name.lower()} during {proc_name.lower()}'
        medical_obj   = 'medical equipment'
    return f'A nurse {action_phrase} using {medical_obj} in a hospital ward.'


# 검증
print('=== GT Caption 예시 ===')
for op_id, action_id in [(5, 8), (6, 20), (8, 21)]:
    cap = build_gt_caption(op_id, action_id)
    print(f'  [{PROCEDURE_MAP[op_id]}] {ACTION_MAP.get(action_id)}')
    print(f'  → {cap}\n')

=== GT Caption 예시 ===
  [Closed Intravenous Infusion] Venipuncture
  → A nurse performing venipuncture on the patient's arm using an IV needle and tourniquet in a hospital ward.

  [Subcutaneous Injection] Perform subcutaneous puncture
  → A nurse performing a subcutaneous puncture on the patient using a syringe and needle in a hospital ward.

  [Blood Glucose Monitoring] Measure blood glucose level
  → A nurse measuring the patient's blood glucose level using a glucometer and test strip in a hospital ward.



### 2-3. 데이터셋 구축

In [7]:
# annotation 로드
with open(ANNO_PATH, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

print(f'총 영상 수: {len(annotations)}')
filtered = {vid: v for vid, v in annotations.items()
            if v['operationID'] in TARGET_PROCEDURES}
print(f'선정 Procedure 영상: {len(filtered)}개')
for name, cnt in Counter(
    PROCEDURE_MAP[v['operationID']] for v in filtered.values()
).most_common():
    print(f'  {name}: {cnt}개')

총 영상 수: 1538
선정 Procedure 영상: 177개
  Closed Intravenous Infusion: 60개
  Subcutaneous Injection: 60개
  Blood Glucose Monitoring: 57개


In [8]:
# frames/{video_id}/ann{N:02d}/frame_XXXX.jpg 구조에서 dataset_items 구성
anno_lookup = {vid: v for vid, v in annotations.items()
               if v['operationID'] in TARGET_PROCEDURES}

dataset_items = []
skip_no_anno = skip_range = skip_tmpl = 0

for fp in sorted(FRAME_DIR.rglob('*.jpg')):
    video_id = fp.parts[-3]
    ann_dir  = fp.parts[-2]

    if video_id not in anno_lookup:
        skip_no_anno += 1
        continue

    ann_idx = int(ann_dir.replace('ann', ''))
    item    = anno_lookup[video_id]
    op_id   = item['operationID']

    if ann_idx >= len(item['annotations']):
        skip_range += 1
        continue

    action_id = item['annotations'][ann_idx]['actionID']

    if (op_id, action_id) not in CAPTION_TEMPLATES:
        skip_tmpl += 1
        continue

    dataset_items.append({
        'image_path': str(fp),
        'caption':    build_gt_caption(op_id, action_id),
        'procedure':  PROCEDURE_MAP[op_id],
        'action':     ACTION_MAP.get(action_id, f'action_{action_id}'),
        'video_id':   video_id,
        'op_id':      op_id,
        'action_id':  action_id,
        'segment':    item['annotations'][ann_idx]['segment'],
    })

print(f'구축된 데이터셋: {len(dataset_items)}개')
print(f'  제외 (annotation 없음): {skip_no_anno} / 인덱스 초과: {skip_range} / template 미정의: {skip_tmpl}')
print()
print('Procedure별:')
for name, cnt in Counter(i['procedure'] for i in dataset_items).most_common():
    print(f'  {name}: {cnt}장')
print('\nAction별 (상위 8):')
for name, cnt in Counter(i['action'] for i in dataset_items).most_common(8):
    print(f'  {name}: {cnt}장')

구축된 데이터셋: 1872개
  제외 (annotation 없음): 188 / 인덱스 초과: 0 / template 미정의: 0

Procedure별:
  Subcutaneous Injection: 886장
  Blood Glucose Monitoring: 607장
  Closed Intravenous Infusion: 379장

Action별 (상위 8):
  Disinfect skin: 294장
  Measure blood glucose level: 229장
  Prepare glucometer: 184장
  Handwashing: 162장
  Inject medication: 161장
  Perform subcutaneous puncture: 138장
  Aspirate medication: 129장
  Release trapped air: 114장


In [9]:
# Video 단위 Train/Val/Test Split (data leakage 방지)
all_video_ids = list(set(i['video_id'] for i in dataset_items))
random.seed(SEED)
random.shuffle(all_video_ids)

n = len(all_video_ids)
train_vids = set(all_video_ids[:int(n * 0.8)])
val_vids   = set(all_video_ids[int(n * 0.8):int(n * 0.9)])
test_vids  = set(all_video_ids[int(n * 0.9):])

train_items = [i for i in dataset_items if i['video_id'] in train_vids]
val_items   = [i for i in dataset_items if i['video_id'] in val_vids]
test_items  = [i for i in dataset_items if i['video_id'] in test_vids]

print(f'Video split: train {len(train_vids)} / val {len(val_vids)} / test {len(test_vids)} videos')
print(f'Frame split: train {len(train_items)} / val {len(val_items)} / test {len(test_items)} frames')

with open(BASE_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_items, f, ensure_ascii=False, indent=2)
print('dataset.json 저장 완료')

Video split: train 116 / val 14 / test 15 videos
Frame split: train 1506 / val 188 / test 178 frames
dataset.json 저장 완료


---
## 3. Metric 계산 유틸리티

In [10]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score


def compute_bleu(predictions, references):
    smoothie = SmoothingFunction().method1
    refs = [[r.lower().split()] for r in references]
    hyps = [p.lower().split() for p in predictions]
    scores = {}
    for n in range(1, 5):
        w = tuple([1.0 / n] * n + [0.0] * (4 - n))
        scores[f'BLEU-{n}'] = corpus_bleu(refs, hyps, weights=w,
                                           smoothing_function=smoothie)
    return scores


def compute_meteor(predictions, references):
    return float(np.mean([
        meteor_score([r.lower().split()], p.lower().split())
        for p, r in zip(predictions, references)
    ]))


def compute_cider(predictions, references):
    try:
        from pycocoevalcap.cider.cider import Cider
        score, _ = Cider().compute_score(
            {i: [r] for i, r in enumerate(references)},
            {i: [p] for i, p in enumerate(predictions)},
        )
        return float(score)
    except Exception as e:
        print(f'CIDEr-D 계산 실패: {e}')
        return 0.0


def compute_all_metrics(predictions, references):
    return {
        **compute_bleu(predictions, references),
        'METEOR':  compute_meteor(predictions, references),
        'CIDEr-D': compute_cider(predictions, references),
    }


print('Metric 함수 정의 완료')

Metric 함수 정의 완료


---
## 4. SmolVLM 로드 및 Caption 생성 함수

In [11]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'

# 과제 요구 4요소: role / action / medical object / hospital context
NURSING_PROMPT = (
    'Describe this medical image in one sentence. '
    'Include: the role (nurse), the specific medical action being performed, '
    'the medical equipment or object being used, '
    'and the hospital context.'
)


def load_smolvlm(model_id=MODEL_ID, device=DEVICE):
    print(f'모델 로드 중: {model_id}')
    processor = AutoProcessor.from_pretrained(model_id)

    # TITAN Xp 12GB 최적화: 이미지 패치 최소화
    if hasattr(processor, 'image_processor'):
        processor.image_processor.size = {'longest_edge': 512}

    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
    ).to(device)
    model.eval()

    if hasattr(model, 'generation_config') and processor.tokenizer.eos_token_id:
        model.generation_config.eos_token_id = processor.tokenizer.eos_token_id
        model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'파라미터: {total:,} total / {trainable:,} trainable')
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated(0) / 1e9
        free = (torch.cuda.get_device_properties(0).total_memory
                - torch.cuda.memory_allocated(0)) / 1e9
        print(f'VRAM 사용: {used:.2f} GB / 여유: {free:.2f} GB')
    return model, processor


def generate_caption(model, processor, image_path,
                     prompt=NURSING_PROMPT, max_new_tokens=64, device=DEVICE):
    image    = Image.open(image_path).convert('RGB')
    messages = [{'role': 'user',
                 'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}]}]
    text     = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs   = processor(text=[text], images=[image],
                         return_tensors='pt').to(device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False)
    generated = out_ids[:, inputs['input_ids'].shape[1]:]
    return processor.decode(generated[0], skip_special_tokens=True).strip()


def generate_captions_batch(model, processor, items,
                             prompt=NURSING_PROMPT, max_new_tokens=64,
                             device=DEVICE):
    preds = []
    for item in tqdm(items, desc='Caption 생성'):
        try:
            path = item['image_path'] if isinstance(item, dict) else str(item)
            preds.append(generate_caption(model, processor, path,
                                          prompt=prompt,
                                          max_new_tokens=max_new_tokens,
                                          device=device))
        except Exception as e:
            print(f'  경고: {e}')
            preds.append('')
    return preds


print('SmolVLM 함수 정의 완료')

/home/undergraduate/20231372_TY/envs/smolvlm_lora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SmolVLM 함수 정의 완료


---
## 5. Baseline: SmolVLM Zero-shot 평가

과제 분석 항목:
- **의료 object 인식 실패**: syringe, IV needle, glucometer 등 미인식
- **의료 행동 collapse**: injecting → "holding something"
- **Hallucination**: 이미지에 없는 의료 용어 등장

In [13]:
# Zero-shot 결과가 이미 있으면 skip (재실행 시 시간 절약)
ZS_RESULT_PATH = MODEL_DIR / 'zero_shot_results.json'

if ZS_RESULT_PATH.exists():
    print('기존 zero-shot 결과 발견 → 로드 후 skip')
    with open(ZS_RESULT_PATH, 'r') as f:
        zs_result = json.load(f)
    zs_metrics     = zs_result['metrics']
    zs_failure     = zs_result['failure']
    zs_predictions = [p['pred'] for p in zs_result['predictions']]
    zs_references  = [p['gt']   for p in zs_result['predictions']]
    eval_test_items = zs_result['predictions']
    print(f'로드 완료: {len(eval_test_items)}개')
    print(f'BLEU-4: {zs_metrics["BLEU-4"]:.4f}  METEOR: {zs_metrics["METEOR"]:.4f}')
else:
    print('Zero-shot 평가 시작...')
    base_model, processor = load_smolvlm(MODEL_ID, DEVICE)

    print(f'\nZero-shot 평가 대상: {len(test_items)}개')
    zs_predictions = generate_captions_batch(
        base_model, processor, test_items, device=DEVICE
    )
    zs_references = [i['caption'] for i in test_items]

    print('\n=== Zero-shot Caption 예시 (첫 5개) ===')
    for i in range(min(5, len(test_items))):
        print(f'\n[{test_items[i]["procedure"]}] {test_items[i]["action"]}')
        print(f'  GT:        {zs_references[i]}')
        print(f'  Zero-shot: {zs_predictions[i]}')

    zs_metrics = compute_all_metrics(zs_predictions, zs_references)
    print('\n=== Zero-shot Metrics ===')
    for k, v in zs_metrics.items():
        print(f'  {k}: {v:.4f}')

    del base_model
    gc.collect()
    torch.cuda.empty_cache()

기존 zero-shot 결과 발견 → 로드 후 skip
로드 완료: 221개
BLEU-4: 0.0047  METEOR: 0.1900


In [14]:
# 실패 분석 함수
MEDICAL_OBJECTS = [
    'syringe', 'needle', 'iv', 'infusion', 'glucometer',
    'glucose', 'test strip', 'lancet', 'tourniquet',
    'catheter', 'swab', 'stethoscope',
]
MEDICAL_ACTIONS = [
    'inject', 'infus', 'venipuncture', 'glucose', 'medication',
    'nurse', 'nursing', 'medical', 'hospital', 'patient',
    'disinfect', 'aspirat', 'puncture', 'monitor',
]
COLLAPSE_WORDS = [
    'holding', 'standing', 'sitting', 'looking',
    'person', 'man', 'woman', 'object', 'something',
]


def analyze_failure(predictions, references):
    obj_fail = action_fail = halluc = 0
    for pred, ref in zip(predictions, references):
        pl, rl = pred.lower(), ref.lower()
        # 1. 의료 object 인식 실패: GT에 있는데 pred에 없음
        if any(o in rl for o in MEDICAL_OBJECTS) and \
           not any(o in pl for o in MEDICAL_OBJECTS):
            obj_fail += 1
        # 2. 의료 행동 collapse: collapse 표현 있고 의료 action 없음
        if any(c in pl for c in COLLAPSE_WORDS) and \
           not any(a in pl for a in MEDICAL_ACTIONS):
            action_fail += 1
        # 3. Hallucination: pred에 GT에 없는 의료 object 등장
        if any(o in pl and o not in rl for o in MEDICAL_OBJECTS):
            halluc += 1
    n = max(len(predictions), 1)
    return {
        'total':                       n,
        'medical_object_fail':         obj_fail,
        'medical_object_fail_pct':     round(obj_fail    / n * 100, 1),
        'medical_action_collapse':     action_fail,
        'medical_action_collapse_pct': round(action_fail / n * 100, 1),
        'hallucination':               halluc,
        'hallucination_pct':           round(halluc       / n * 100, 1),
    }


if not ZS_RESULT_PATH.exists():
    zs_failure = analyze_failure(zs_predictions, zs_references)
    print('=== Zero-shot 실패 분석 ===')
    print(f'  의료 Object 인식 실패: {zs_failure["medical_object_fail_pct"]}%')
    print(f'  의료 행동 Collapse:    {zs_failure["medical_action_collapse_pct"]}%')
    print(f'  Hallucination:         {zs_failure["hallucination_pct"]}%')

    eval_test_items = [
        {'image_path': item['image_path'], 'gt': ref, 'pred': pred,
         'procedure': item['procedure'], 'action': item['action'],
         'op_id': item['op_id'], 'action_id': item['action_id']}
        for item, ref, pred in zip(test_items, zs_references, zs_predictions)
    ]
    zs_result = {'metrics': zs_metrics, 'failure': zs_failure,
                 'predictions': eval_test_items}
    with open(ZS_RESULT_PATH, 'w', encoding='utf-8') as f:
        json.dump(zs_result, f, ensure_ascii=False, indent=2)
    print('Zero-shot 결과 저장 완료')
else:
    zs_failure = zs_result['failure']

print('=== Zero-shot 실패 분석 ===')
print(f'  의료 Object 인식 실패: {zs_failure["medical_object_fail_pct"]}%')
print(f'  의료 행동 Collapse:    {zs_failure["medical_action_collapse_pct"]}%')
print(f'  Hallucination:         {zs_failure["hallucination_pct"]}%')

=== Zero-shot 실패 분석 ===
  의료 Object 인식 실패: 74.7%
  의료 행동 Collapse:    0.9%
  Hallucination:         14.5%


---
## 6. LoRA Fine-tuning

### 6-1. Dataset / Collate 정의

In [15]:
from torch.utils.data import Dataset, DataLoader


class NursingCaptionDataset(Dataset):
    def __init__(self, items, processor, prompt=NURSING_PROMPT):
        self.items     = items
        self.processor = processor
        self.prompt    = prompt

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item  = self.items[idx]
        image = Image.open(item['image_path']).convert('RGB')
        messages = [
            {'role': 'user',
             'content': [{'type': 'image'}, {'type': 'text', 'text': self.prompt}]},
            {'role': 'assistant',
             'content': [{'type': 'text', 'text': item['caption']}]},
        ]
        text = self.processor.apply_chat_template(
            messages, add_generation_prompt=False
        )
        return {'image': image, 'text': text, 'caption': item['caption']}


def nursing_collate_fn(batch, processor, device):
    images   = [b['image']   for b in batch]
    texts    = [b['text']    for b in batch]
    captions = [b['caption'] for b in batch]
    inputs = processor(
        text=texts, images=images,
        return_tensors='pt', padding=True, truncation=False,
    ).to(device)
    labels = inputs['input_ids'].clone()
    for i, caption in enumerate(captions):
        cap_ids    = processor.tokenizer(
            caption, add_special_tokens=False
        )['input_ids']
        prompt_len = labels.shape[1] - len(cap_ids)
        if prompt_len > 0:
            labels[i, :prompt_len] = -100   # prompt 구간 loss 제외
    labels[inputs['attention_mask'] == 0] = -100
    inputs['labels'] = labels
    return inputs


print('Dataset / Collate 정의 완료')

Dataset / Collate 정의 완료


### 6-2. LoRA 설정 (TITAN Xp 12GB 최적화)

In [16]:
from peft import get_peft_model, LoraConfig, TaskType


@dataclass
class LoRAConfig:
    model_id:       str   = MODEL_ID
    rank:           int   = 16          # LoRA rank
    alpha:          int   = 32          # LoRA scaling
    dropout:        float = 0.05
    target_modules: tuple = (
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    )
    # TITAN Xp 12GB 최적화 설정
    num_epochs:     int   = 3
    batch_size:     int   = 4
    grad_accum:     int   = 4           # 실효 batch = 8 (16→8: 속도↑)
    lr:             float = 2e-4
    weight_decay:   float = 0.01
    max_new_tokens: int   = 64
    output_dir:     Path  = MODEL_DIR


lora_cfg = LoRAConfig()
print(f'rank={lora_cfg.rank}, alpha={lora_cfg.alpha}')
print(f'lr={lora_cfg.lr}, epochs={lora_cfg.num_epochs}')
print(f'실효 batch = {lora_cfg.batch_size} × {lora_cfg.grad_accum} = {lora_cfg.batch_size * lora_cfg.grad_accum}')

rank=16, alpha=32
lr=0.0002, epochs=3
실효 batch = 4 × 4 = 16


### 6-3. 모델 로드 + LoRA 적용

In [17]:
ft_model, ft_processor = load_smolvlm(lora_cfg.model_id, DEVICE)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=lora_cfg.rank,
    lora_alpha=lora_cfg.alpha,
    lora_dropout=lora_cfg.dropout,
    target_modules=list(lora_cfg.target_modules),
    bias='none',
    inference_mode=False,
)
ft_model = get_peft_model(ft_model, peft_config)
ft_model.print_trainable_parameters()

모델 로드 중: HuggingFaceTB/SmolVLM-256M-Instruct


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 4250.38it/s]


파라미터: 256,484,928 total / 256,484,928 trainable
VRAM 사용: 0.51 GB / 여유: 12.27 GB
trainable params: 5,769,216 || all params: 262,254,144 || trainable%: 2.1999


### 6-4. 학습 루프

In [18]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

collate_fn    = partial(nursing_collate_fn, processor=ft_processor, device=DEVICE)
train_dataset = NursingCaptionDataset(train_items, ft_processor)
val_dataset   = NursingCaptionDataset(val_items,   ft_processor)

train_loader = DataLoader(train_dataset, batch_size=lora_cfg.batch_size,
                          shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=lora_cfg.batch_size,
                          shuffle=False, collate_fn=collate_fn, num_workers=0)

optimizer   = AdamW(
    [p for p in ft_model.parameters() if p.requires_grad],
    lr=lora_cfg.lr, weight_decay=lora_cfg.weight_decay
)
total_steps = len(train_loader) * lora_cfg.num_epochs // lora_cfg.grad_accum
scheduler   = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

print(f'Train {len(train_loader)} steps / Val {len(val_loader)} steps')
print(f'총 최적화 스텝: {total_steps}')

Train 377 steps / Val 47 steps
총 최적화 스텝: 282


In [19]:
import time

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(lora_cfg.num_epochs):
    epoch_start = time.time()

    # Train
    ft_model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f'Epoch {epoch+1}/{lora_cfg.num_epochs} [Train]')

    for step, batch in pbar:
        outputs    = ft_model(**batch)
        loss       = outputs.loss / lora_cfg.grad_accum
        loss.backward()
        epoch_loss += loss.item() * lora_cfg.grad_accum

        if (step + 1) % lora_cfg.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if torch.cuda.is_available():
            vram = torch.cuda.memory_allocated(0) / 1e9
            pbar.set_postfix({
                'loss': f'{epoch_loss / (step + 1):.4f}',
                'vram': f'{vram:.1f}G'
            })

    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # Validation
    ft_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]'):
            val_loss += ft_model(**batch).loss.item()
    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)

    elapsed = (time.time() - epoch_start) / 60
    print(f'Epoch {epoch+1}: train={avg_train:.4f}  val={avg_val:.4f}  ({elapsed:.1f}분)')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        ft_model.save_pretrained(lora_cfg.output_dir / 'best_lora')
        ft_processor.save_pretrained(lora_cfg.output_dir / 'best_lora')
        print(f'  → Best 저장 (val={best_val_loss:.4f})')

print('\n학습 완료!')

Epoch 1 [Val]: 100%|██████████| 47/47 [00:18<00:00,  2.54it/s]


Epoch 1: train=0.6053  val=0.1501  (4.9분)
  → Best 저장 (val=0.1501)


Epoch 2 [Val]: 100%|██████████| 47/47 [00:18<00:00,  2.54it/s]


Epoch 2: train=0.0566  val=0.1369  (5.0분)
  → Best 저장 (val=0.1369)


Epoch 3 [Val]: 100%|██████████| 47/47 [00:18<00:00,  2.49it/s]


Epoch 3: train=0.0398  val=0.1259  (5.0분)
  → Best 저장 (val=0.1259)

학습 완료!


In [20]:
# Loss 곡선 저장
plt.figure(figsize=(8, 4))
plt.plot(range(1, lora_cfg.num_epochs + 1), train_losses, 'b-o', label='Train Loss')
plt.plot(range(1, lora_cfg.num_epochs + 1), val_losses,   'r-o', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LoRA Fine-tuning Loss Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(lora_cfg.output_dir / 'loss_curve.png', dpi=150)
plt.close()
print(f'Loss 곡선 저장: {lora_cfg.output_dir / "loss_curve.png"}')

# GPU 정리
del ft_model
gc.collect()
torch.cuda.empty_cache()
print('GPU 메모리 정리 완료')

Loss 곡선 저장: /data/20221373_YY/nurse/outputs/model/loss_curve.png
GPU 메모리 정리 완료


---
## 7. Fine-tuned 모델 평가

In [21]:
from peft import PeftModel

base_for_eval, eval_processor = load_smolvlm(lora_cfg.model_id, DEVICE)
ft_model_eval = PeftModel.from_pretrained(
    base_for_eval, lora_cfg.output_dir / 'best_lora'
)
ft_model_eval.eval()
print('Fine-tuned 모델 로드 완료')

# zero-shot과 완전히 동일한 이미지로 평가 (정성 비교를 위해 필수)
ft_predictions = generate_captions_batch(
    ft_model_eval, eval_processor, eval_test_items,
    max_new_tokens=lora_cfg.max_new_tokens, device=DEVICE
)

ft_metrics = compute_all_metrics(ft_predictions, zs_references)
ft_failure = analyze_failure(ft_predictions, zs_references)

print('\n=== Fine-tuned Metrics ===')
for k, v in ft_metrics.items():
    print(f'  {k}: {v:.4f}')

모델 로드 중: HuggingFaceTB/SmolVLM-256M-Instruct


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 3935.36it/s]


파라미터: 256,484,928 total / 256,484,928 trainable
VRAM 사용: 0.69 GB / 여유: 12.10 GB
Fine-tuned 모델 로드 완료


Caption 생성: 100%|██████████| 221/221 [03:56<00:00,  1.07s/it]



=== Fine-tuned Metrics ===
  BLEU-1: 0.7066
  BLEU-2: 0.6251
  BLEU-3: 0.5608
  BLEU-4: 0.5181
  METEOR: 0.7147
  CIDEr-D: 3.7799


---
## 8. Before / After 비교 분석

### 8-1. 정량 평가 테이블

In [22]:
METRIC_KEYS = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'METEOR', 'CIDEr-D']

print('=' * 62)
print(f'{"Metric":<12} {"Zero-shot":>12} {"LoRA FT":>12} {"향상(Δ)":>12}')
print('=' * 62)
for m in METRIC_KEYS:
    zv   = zs_metrics.get(m, 0)
    fv   = ft_metrics.get(m, 0)
    diff = fv - zv
    sign = '+' if diff >= 0 else ''
    print(f'{m:<12} {zv:>12.4f} {fv:>12.4f} {sign}{diff:>11.4f}')
print('=' * 62)

print('\n=== 실패 분석 비교 ===')
print(f'{"항목":<30} {"Zero-shot":>10} {"LoRA FT":>10} {"Δ":>8}')
print('-' * 60)
for key, label in [
    ('medical_object_fail_pct',     '의료 Object 인식 실패(%)'),
    ('medical_action_collapse_pct', '의료 행동 Collapse(%)'),
    ('hallucination_pct',           'Hallucination(%)'),
]:
    zv   = zs_failure[key]
    fv   = ft_failure[key]
    diff = fv - zv
    sign = '+' if diff >= 0 else ''
    print(f'{label:<30} {zv:>9.1f}% {fv:>9.1f}%  ({sign}{diff:.1f}%)')

Metric          Zero-shot      LoRA FT        향상(Δ)
BLEU-1             0.2833       0.7066 +     0.4233
BLEU-2             0.1382       0.6251 +     0.4869
BLEU-3             0.0255       0.5608 +     0.5353
BLEU-4             0.0047       0.5181 +     0.5134
METEOR             0.1900       0.7147 +     0.5247
CIDEr-D            0.0280       3.7799 +     3.7519

=== 실패 분석 비교 ===
항목                              Zero-shot    LoRA FT        Δ
------------------------------------------------------------
의료 Object 인식 실패(%)                  74.7%       0.0%  (-74.7%)
의료 행동 Collapse(%)                    0.9%       0.0%  (-0.9%)
Hallucination(%)                    14.5%      57.5%  (+43.0%)


### 8-2. 정성 평가: 동일 이미지 Before / After 시각화

**과제 요구사항**: 동일 이미지에 대해 SmolVLM 결과 vs LoRA Fine-tuning 결과 비교

각 Procedure별 핵심 action 1개씩 선정 → 실제 이미지 + GT + Zero-shot + LoRA FT 한 화면 표시

In [23]:
# Procedure별 핵심 action 우선 선정
PRIORITY_ACTIONS = {
    'Closed Intravenous Infusion': 'Venipuncture',
    'Subcutaneous Injection':      'Perform subcutaneous puncture',
    'Blood Glucose Monitoring':    'Measure blood glucose level',
}

selected_indices = []
used_procs = set()

# 1순위: 핵심 action
for idx, item in enumerate(eval_test_items):
    if item['procedure'] not in used_procs and \
       item['action'] == PRIORITY_ACTIONS.get(item['procedure'], ''):
        selected_indices.append(idx)
        used_procs.add(item['procedure'])

# 2순위: 미채운 procedure 채우기
for idx, item in enumerate(eval_test_items):
    if len(selected_indices) >= 3:
        break
    if item['procedure'] not in used_procs:
        selected_indices.append(idx)
        used_procs.add(item['procedure'])

# 3순위: 추가 샘플 (총 4개 확보)
for idx in range(len(eval_test_items)):
    if len(selected_indices) >= 4:
        break
    if idx not in selected_indices:
        selected_indices.append(idx)

N_COMPARE = len(selected_indices)
print(f'정성 평가 선정: {N_COMPARE}장')
for idx in selected_indices:
    item = eval_test_items[idx]
    print(f'  [{item["procedure"]}] {item["action"]}')

정성 평가 선정: 4장
  [Subcutaneous Injection] Perform subcutaneous puncture
  [Blood Glucose Monitoring] Measure blood glucose level
  [Closed Intravenous Infusion] Venipuncture
  [Closed Intravenous Infusion] Connect infusion device


In [24]:
# ── 동일 이미지 Before/After 시각화 ──────────────────────────────────────
# 컬럼 구성: [실제 이미지 | 텍스트 비교 (GT / Zero-shot / LoRA FT)]

fig, axes = plt.subplots(
    N_COMPARE, 2,
    figsize=(18, 6 * N_COMPARE),
    gridspec_kw={'width_ratios': [1, 2.5]}
)
if N_COMPARE == 1:
    axes = [axes]

BG_COLORS = {'GT': '#d5f5e3', 'ZS': '#fde8e8', 'FT': '#d6eaf8'}
LABELS    = {'GT': 'GT Caption', 'ZS': 'Zero-shot (SmolVLM)', 'FT': 'LoRA Fine-tuned'}

for row, sample_idx in enumerate(selected_indices):
    item    = eval_test_items[sample_idx]
    gt      = zs_references[sample_idx]
    zs_pred = zs_predictions[sample_idx]
    ft_pred = ft_predictions[sample_idx]

    ax_img  = axes[row][0]
    ax_text = axes[row][1]

    # 이미지
    try:
        img = Image.open(item['image_path']).convert('RGB')
        ax_img.imshow(img)
        ax_img.axis('off')
    except Exception:
        ax_img.text(0.5, 0.5, 'Image Not Found',
                    ha='center', va='center', transform=ax_img.transAxes,
                    fontsize=11, color='red')
        ax_img.axis('off')

    ax_img.set_title(
        f'[{item["procedure"]}]\n{item["action"]}',
        fontsize=9, fontweight='bold', pad=6
    )

    # 텍스트 비교: 3개 박스로 분리 표시
    ax_text.axis('off')
    y_positions = [0.95, 0.62, 0.29]   # GT / Zero-shot / LoRA FT
    texts_info  = [
        ('GT',  gt,      BG_COLORS['GT']),
        ('ZS',  zs_pred, BG_COLORS['ZS']),
        ('FT',  ft_pred, BG_COLORS['FT']),
    ]
    for (key, text, bg), y in zip(texts_info, y_positions):
        label = LABELS[key]
        # 라벨
        ax_text.text(
            0.02, y, f'▶ {label}',
            transform=ax_text.transAxes,
            fontsize=9, fontweight='bold', va='top',
            color={'GT': '#1e8449', 'ZS': '#c0392b', 'FT': '#1a5276'}[key]
        )
        # 내용 박스
        ax_text.text(
            0.02, y - 0.05, text,
            transform=ax_text.transAxes,
            fontsize=8.5, va='top', wrap=True,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=bg, alpha=0.7,
                      edgecolor='gray', linewidth=0.5)
        )

plt.suptitle(
    'Before / After Caption 비교\n'
    'SmolVLM Zero-shot  vs  LoRA Fine-tuned  (동일 이미지)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout(pad=1.5)
save_path = lora_cfg.output_dir / 'before_after_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Before/After 비교 저장: {save_path}')

/tmp/ipykernel_2972355/957175039.py:71: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(pad=1.5)
/tmp/ipykernel_2972355/957175039.py:71: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(pad=1.5)
/tmp/ipykernel_2972355/957175039.py:71: UserWarning: Glyph 46041 (\N{HANGUL SYLLABLE DONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout(pad=1.5)
/tmp/ipykernel_2972355/957175039.py:71: UserWarning: Glyph 51068 (\N{HANGUL SYLLABLE IL}) missing from font(s) DejaVu Sans.
  plt.tight_layout(pad=1.5)
/tmp/ipykernel_2972355/957175039.py:71: UserWarning: Glyph 51060 (\N{HANGUL SYLLABLE I}) missing from font(s) DejaVu Sans.
  plt.tight_layout(pad=1.5)
/tmp/ipykernel_2972355/957175039.py:71: UserWarning: Glyph 48120 (\N{HANGUL SYLLABLE MI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(pad=1.5)
/tmp/ipykernel_2972355/957175039.py:71: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE J

Before/After 비교 저장: /data/20221373_YY/nurse/outputs/model/before_after_comparison.png


### 8-3. Metric Bar Chart

In [25]:
metrics_to_plot = ['BLEU-1', 'BLEU-2', 'BLEU-4', 'METEOR', 'CIDEr-D']
zs_vals = [zs_metrics.get(m, 0) for m in metrics_to_plot]
ft_vals = [ft_metrics.get(m, 0) for m in metrics_to_plot]

x, w = np.arange(len(metrics_to_plot)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w / 2, zs_vals, w,
            label='Zero-shot (SmolVLM)', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w / 2, ft_vals, w,
            label='LoRA Fine-tuned',     color='tomato',    alpha=0.85)

for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel('Score')
ax.set_title('SmolVLM Zero-shot vs LoRA Fine-tuned\nNursing Domain Caption Metrics')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(lora_cfg.output_dir / 'metric_comparison.png', dpi=150)
plt.close()
print(f'Metric 차트 저장: {lora_cfg.output_dir / "metric_comparison.png"}')

Metric 차트 저장: /data/20221373_YY/nurse/outputs/model/metric_comparison.png


### 8-4. 최종 결과 저장

In [26]:
final_results = {
    'dataset': {
        'total': len(dataset_items),
        'train': len(train_items),
        'val':   len(val_items),
        'test':  len(test_items),
        'procedures': list(PROCEDURE_MAP.values()),
        'split_unit': 'video (leakage-free)',
    },
    'model': lora_cfg.model_id,
    'lora_config': {
        'rank':                 lora_cfg.rank,
        'alpha':                lora_cfg.alpha,
        'dropout':              lora_cfg.dropout,
        'target_modules':       list(lora_cfg.target_modules),
        'num_epochs':           lora_cfg.num_epochs,
        'lr':                   lora_cfg.lr,
        'effective_batch_size': lora_cfg.batch_size * lora_cfg.grad_accum,
    },
    'zero_shot_metrics': zs_metrics,
    'lora_ft_metrics':   ft_metrics,
    'improvement': {
        m: round(ft_metrics.get(m, 0) - zs_metrics.get(m, 0), 4)
        for m in METRIC_KEYS
    },
    'failure_analysis': {
        'zero_shot': zs_failure,
        'lora_ft':   ft_failure,
    },
    # 정성 평가 샘플 (동일 이미지 GT / Zero-shot / LoRA FT)
    'qualitative_samples': [
        {
            'procedure': eval_test_items[idx]['procedure'],
            'action':    eval_test_items[idx]['action'],
            'gt':        zs_references[idx],
            'zero_shot': zs_predictions[idx],
            'lora_ft':   ft_predictions[idx],
        }
        for idx in selected_indices
    ],
    # 전체 test 샘플
    'all_test_samples': [
        {
            'procedure': eval_test_items[i]['procedure'],
            'action':    eval_test_items[i]['action'],
            'gt':        zs_references[i],
            'zero_shot': zs_predictions[i],
            'lora_ft':   ft_predictions[i],
        }
        for i in range(len(eval_test_items))
    ],
    'train_losses': train_losses,
    'val_losses':   val_losses,
}

with open(lora_cfg.output_dir / 'final_results.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print('=' * 65)
print('최종 요약')
print('=' * 65)
for m in ['BLEU-4', 'METEOR', 'CIDEr-D']:
    zv   = zs_metrics[m]
    fv   = ft_metrics[m]
    dv   = final_results['improvement'][m]
    sign = '+' if dv >= 0 else ''
    print(f'{m:<10}  Zero-shot: {zv:.4f}  →  LoRA FT: {fv:.4f}  (Δ{sign}{dv:.4f})')
print()
print('저장 파일 목록:')
for fname in [
    'zero_shot_results.json',
    'final_results.json',
    'loss_curve.png',
    'metric_comparison.png',
    'before_after_comparison.png',
    'best_lora/',
]:
    path   = lora_cfg.output_dir / fname
    exists = '✅' if path.exists() else '❌'
    print(f'  {exists} {path}')

최종 요약
BLEU-4      Zero-shot: 0.0047  →  LoRA FT: 0.5181  (Δ+0.5134)
METEOR      Zero-shot: 0.1900  →  LoRA FT: 0.7147  (Δ+0.5247)
CIDEr-D     Zero-shot: 0.0280  →  LoRA FT: 3.7799  (Δ+3.7519)

저장 파일 목록:
  ✅ /data/20221373_YY/nurse/outputs/model/zero_shot_results.json
  ✅ /data/20221373_YY/nurse/outputs/model/final_results.json
  ✅ /data/20221373_YY/nurse/outputs/model/loss_curve.png
  ✅ /data/20221373_YY/nurse/outputs/model/metric_comparison.png
  ✅ /data/20221373_YY/nurse/outputs/model/before_after_comparison.png
  ✅ /data/20221373_YY/nurse/outputs/model/best_lora
